# Ablation 1 Diagnostic: Focal Tversky Loss - Ultra-Light Validation

**Purpose**: Validate production training workflow with Focal Tversky Loss on minimal dataset

**Strategy**: Same workflow as production, much smaller:
- **3 training images** (vs 2,208)
- **1 seed** (42) (vs 3 seeds)
- **5 epochs** (vs 100)
- **Focal Tversky Loss** (α=0.7, β=0.3, γ=2.0)

**Runtime**: ~3-4 minutes total
**Validates**: Loss function integrates correctly, training converges


In [ ]:
# Fix: Lock NumPy at Kaggle's default (prevent 2.x upgrade from ultralytics)
import sys

print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np

default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess

subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default (prevent 2.x upgrade)...")
subprocess.run(
    ["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"],
    check=True,
)

print("\nInstallation complete!")
print("\nVerifying versions...")
import numpy as np

print(f"   NumPy: {np.__version__}")

from ultralytics import YOLO

print("   OK Ultralytics imported successfully")


In [ ]:
import json
import os
import sys
from pathlib import Path
import time

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Add code dataset to path
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Code dataset added to Python path")
    # Import modules
    from data.preprocessing import convert_mask_to_yolo_format
    from data.yolo_config import create_yolo_config, validate_yolo_config
    from models.focal_tversky import FocalTverskyLoss
    from evaluation.metrics import calculate_segmentation_metrics, calculate_boundary_f1
    from evaluation.yolo_utils import yolo_result_to_metrics_input
else:
    print("[ERROR] Code dataset not found - diagnostic must run on Kaggle")
    raise FileNotFoundError("/kaggle/input/wound-seg-code-v1")

print("OK All modules imported successfully")


In [ ]:
# Environment (Kaggle-only for diagnostic)
BASE_INPUT_PATH = Path("/kaggle/input/baseline-outputs")
OUTPUT_PATH = Path("/kaggle/working")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: Kaggle Diagnostic")
print(f"Input path: {BASE_INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")


In [ ]:
# Copy minimal subset from baseline-outputs (use test images as "training" data)
import shutil

print("Copying minimal dataset (3 test images for diagnostic)...")

# Create destination
DIAGNOSTIC_DATASET = OUTPUT_PATH / "diagnostic_data"
DIAGNOSTIC_DATASET.mkdir(exist_ok=True)

# Directories
train_images_dir = DIAGNOSTIC_DATASET / "train_images"
train_masks_dir = DIAGNOSTIC_DATASET / "train_masks"

train_images_dir.mkdir(exist_ok=True)
train_masks_dir.mkdir(exist_ok=True)

# Source paths (NOTE: baseline-outputs has test/ not train/, so use test images)
source_images = BASE_INPUT_PATH / "data_wound_seg/test/images"
source_masks = BASE_INPUT_PATH / "data_wound_seg/test/labels"

# Copy first 3 test images (use as diagnostic "training" data)
all_images = sorted(source_images.glob("*.png"))[:3]

for img_path in all_images:
    shutil.copy(img_path, train_images_dir / img_path.name)
    
    # Labels are .txt files in YOLO format, not .png masks!
    # For diagnostic, we'll need to skip this and use wound-segmentation-images instead
    label_path = source_masks / f"{img_path.stem}.txt"
    if label_path.exists():
        # This won't work - need actual masks, not labels
        pass

# Actually, baseline-outputs doesn't have masks - need different dataset
# Let's use wound-segmentation-images but ONLY copy 3 files

source_images_full = Path("/kaggle/input/wound-segmentation-images/data_wound_seg/train_images")
source_masks_full = Path("/kaggle/input/wound-segmentation-images/data_wound_seg/train_masks")

if source_images_full.exists():
    # Use full dataset, but only 3 images
    all_images = sorted(source_images_full.glob("*.png"))[:3]
    
    for img_path in all_images:
        shutil.copy(img_path, train_images_dir / img_path.name)
        
        mask_path = source_masks_full / img_path.name
        if mask_path.exists():
            shutil.copy(mask_path, train_masks_dir / img_path.name)

n_images = len(list(train_images_dir.glob("*.png")))
n_masks = len(list(train_masks_dir.glob("*.png")))

print(f"OK Copied {n_images} images, {n_masks} masks")
print(f"\nDiagnostic dataset: {n_images} images (ultra-light for speed)")


In [ ]:
# Create YOLO directory structure and convert labels
print("Creating YOLO dataset structure...")

# Create train/ directory with images/ and labels/
train_yolo_dir = DIAGNOSTIC_DATASET / "train"
(train_yolo_dir / "images").mkdir(parents=True, exist_ok=True)
(train_yolo_dir / "labels").mkdir(parents=True, exist_ok=True)

# Copy images to YOLO structure
for img_path in train_images_dir.glob("*.png"):
    dest = train_yolo_dir / "images" / img_path.name
    shutil.copy(img_path, dest)

n_images = len(list((train_yolo_dir / "images").glob("*.png")))
print(f"OK {n_images} images in YOLO structure")

# Convert masks to YOLO labels
print("\nConverting masks to YOLO label format...")

labels_dir = train_yolo_dir / "labels"
converted_count = 0

for mask_path in train_masks_dir.glob("*.png"):
    label_path = labels_dir / f"{mask_path.stem}.txt"
    
    result = convert_mask_to_yolo_format(
        mask_path=str(mask_path),
        image_width=512,
        image_height=512,
        output_path=str(label_path)
    )
    
    if result["success"]:
        converted_count += 1

print(f"OK Converted {converted_count} masks to YOLO labels")

n_labels = len(list(labels_dir.glob("*.txt")))
print(f"OK Verification: {n_labels} label files created")

if n_labels == 0:
    raise ValueError("Label conversion failed - no label files created")


In [ ]:
# Create YOLO configuration
print("\nCreating YOLO configuration...")

config_result = create_yolo_config(
    dataset_root=DIAGNOSTIC_DATASET,
    train_dir="train",
    val_dir="train",  # Use same for diagnostic (no validation split)
    test_dir="train"
)

if not config_result["success"]:
    raise ValueError(f"Config creation failed: {config_result['error']}")

config_path = config_result["data"]["config_path"]
print(f"OK Configuration created: {config_path}")

# Validate
validation_result = validate_yolo_config(config_path)
if validation_result["success"]:
    print(f"OK Configuration validated")
    print(f"  Dataset root: {validation_result['data']['path']}")
    print(f"  Classes: {validation_result['data']['nc']}")
else:
    raise ValueError(f"Validation failed: {validation_result['error']}")


## Custom Trainer with Focal Tversky Loss

**Integration**: Modify YOLOv8's default segmentation loss to use Focal Tversky Loss.

**Loss Parameters**:
- α = 0.7 (penalize false negatives more - clinical preference)
- β = 0.3 (complements α)
- γ = 2.0 (focal parameter - emphasize hard examples)


In [ ]:
# Create custom trainer with Focal Tversky Loss
from ultralytics.models.yolo.segment import SegmentationTrainer
from ultralytics.utils import ops

class FocalTverskyTrainer(SegmentationTrainer):
    """Custom YOLOv8 trainer using Focal Tversky Loss for segmentation."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Initialize Focal Tversky Loss
        self.focal_tversky_loss = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=2.0)
    
    def criterion(self, preds, batch):
        """Override loss calculation to use Focal Tversky Loss."""
        # Call parent to get default loss components
        loss = super().criterion(preds, batch)
        
        # Replace segmentation loss with Focal Tversky Loss
        if 'seg' in preds:
            # Extract predicted masks and targets
            pred_masks = preds['seg']  # Shape: (batch, 1, H, W)
            
            # Get target masks from batch
            if 'masks' in batch:
                target_masks = batch['masks'].float()
                
                # Apply sigmoid to predictions if needed
                if pred_masks.min() < 0 or pred_masks.max() > 1:
                    pred_masks = torch.sigmoid(pred_masks)
                
                # Calculate Focal Tversky Loss
                seg_loss = self.focal_tversky_loss(pred_masks, target_masks)
                
                # Replace default segmentation loss
                loss[2] = seg_loss  # loss[2] is segmentation loss component
        
        return loss

print("OK FocalTverskyTrainer defined")
print(f"  Loss function: Focal Tversky Loss (α=0.7, β=0.3, γ=2.0)")


## Diagnostic Training Run

**Configuration**:
- **1 seed** (42)
- **5 epochs** (enough to see the loss trend)
- **Batch 2** (tiny dataset)
- **Focal Tversky Loss**

**Expected**: Loss should decrease, no errors


In [ ]:
# Diagnostic training with Focal Tversky Loss
print("=" * 80)
print("STARTING DIAGNOSTIC TRAINING")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Seed: 42")
print(f"  Epochs: 5")
print(f"  Batch: 2")
print(f"  Loss: Focal Tversky (α=0.7, β=0.3, γ=2.0)")
print(f"  Images: 3")
print(f"\nDataset config: {config_path}\n")

# Initialize model
model = YOLO("yolov8n-seg.pt")

# Replace trainer with custom Focal Tversky trainer
model.trainer = FocalTverskyTrainer

# Train
training_start = time.time()

try:
    results = model.train(
        data=str(config_path),
        epochs=5,
        batch=2,
        imgsz=512,
        patience=3,
        seed=42,
        save=True,
        project=str(OUTPUT_PATH / "diagnostic_training"),
        name="focal_tversky_test",
        exist_ok=True,
        verbose=False,
        plots=False  # Skip plots for speed
    )
    
    training_duration = time.time() - training_start
    training_success = True
    
    print(f"\nOK Training completed in {training_duration:.1f} seconds")
    
except Exception as e:
    training_duration = time.time() - training_start
    training_success = False
    print(f"\nFAIL Training failed after {training_duration:.1f} seconds")
    print(f"   Error: {e}")
    raise


In [ ]:
# Analyze training results
if training_success:
    results_path = OUTPUT_PATH / "diagnostic_training/focal_tversky_test/results.csv"
    
    if results_path.exists():
        import pandas as pd
        metrics_df = pd.read_csv(results_path)
        metrics_df.columns = metrics_df.columns.str.strip()
        
        print("\nTraining Metrics Summary:")
        print(f"  Epochs completed: {len(metrics_df)}")
        
        # Check loss trend
        if 'train/seg_loss' in metrics_df.columns:
            initial_loss = metrics_df['train/seg_loss'].iloc[0]
            final_loss = metrics_df['train/seg_loss'].iloc[-1]
            loss_reduction = ((initial_loss - final_loss) / initial_loss) * 100
            
            print(f"  Initial seg loss: {initial_loss:.4f}")
            print(f"  Final seg loss: {final_loss:.4f}")
            print(f"  Loss reduction: {loss_reduction:.1f}%")
            
            loss_decreased = final_loss < initial_loss
            print(f"  OK Loss decreased: {loss_decreased}")
        
        # Display epoch-by-epoch
        if len(metrics_df) <= 10:
            print("\nEpoch-by-Epoch Metrics:")
            print(metrics_df[['epoch', 'train/seg_loss']].to_string(index=False))
    else:
        print(f"WARNING Results file not found: {results_path}")
else:
    print("WARNING Training failed - skipping analysis")


In [ ]:
# Export diagnostic report
diagnostic_report = {
    "notebook": "ablation_1_diagnostic",
    "training_success": training_success,
    "training_time_seconds": training_duration,
    "configuration": {
        "loss_function": "FocalTverskyLoss",
        "alpha": 0.7,
        "beta": 0.3,
        "gamma": 2.0,
        "epochs": 5,
        "batch_size": 2,
        "image_size": 512,
        "n_training_images": 3,
        "seed": 42
    },
    "environment": {
        "pytorch_version": "Kaggle default",
        "cuda_available": False,
        "device": "cpu"
    }
}

# Add metrics if successful
if training_success and results_path.exists():
    metrics_df = pd.read_csv(results_path)
    metrics_df.columns = metrics_df.columns.str.strip()
    
    if 'train/seg_loss' in metrics_df.columns:
        diagnostic_report["metrics"] = {
            "initial_loss": float(metrics_df['train/seg_loss'].iloc[0]),
            "final_loss": float(metrics_df['train/seg_loss'].iloc[-1]),
            "loss_reduction_percent": float(
                ((metrics_df['train/seg_loss'].iloc[0] - metrics_df['train/seg_loss'].iloc[-1]) 
                 / metrics_df['train/seg_loss'].iloc[0]) * 100
            ),
            "epochs_completed": len(metrics_df)
        }

# Save report
report_path = OUTPUT_PATH / "ablation_1_diagnostic_report.json"
with open(report_path, 'w') as f:
    json.dump(diagnostic_report, f, indent=2)

print(f"OK Diagnostic report saved to: {report_path}")
print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

if training_success:
    print("\nOK SUCCESS: Focal Tversky Loss integrated correctly")
    print("\nNext step: Deploy full production training (2,208 images, 3 seeds)")
else:
    print("\nFAIL FAILED: Check error messages above")
    print("\nFix issues before proceeding to production")
